# E04 Breast Cancer：缩放、PCA 与召回权衡

> Issue #9 / Epic #6 / 路线图 E04

## 你将学到
1. 特征量纲差异大 → 必须缩放再进线性/距离模型
2. PCA 降维可加速，但可解释性下降
3. 医疗场景：**漏诊成本高**，不能只看 Accuracy

## 指标重点
- Recall（召回）/ Precision-Recall 曲线
- 阈值扫描：用业务成本选阈值，而不是默认 0.5


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载 breast_cancer


In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
print(df.shape)
print(df['target'].value_counts())  # 1=malignant? 以 data.target_names 为准
print('target_names:', list(data.target_names))
df.head()


## 2. EDA：量纲差异


In [ ]:
from wanglaoshi import DataAnalyzer

analyzer = DataAnalyzer(df)
display(analyzer.basic_statistics().head(12))
print('feature scale span (max/min of mean):',
      df.drop(columns=['target']).mean().max() / max(df.drop(columns=['target']).mean().min(), 1e-9))


## 3. Pipeline：StandardScaler + PCA + LogisticRegression


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

X = df.drop(columns=['target'])
y = df['target']
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10 if QUICK else 15, random_state=RANDOM_STATE)),
    ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
pipe.fit(Xtr, ytr)
proba = pipe.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)
print(classification_report(yte, pred, target_names=list(data.target_names)))
print('ROC-AUC', roc_auc_score(yte, proba))
print('AP', average_precision_score(yte, proba))


## 4. 阈值扫描：提高召回


In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score

rows = []
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (proba >= thr).astype(int)
    rows.append({
        'thr': thr,
        'precision': precision_score(yte, p, zero_division=0),
        'recall': recall_score(yte, p, zero_division=0),
        'f1': f1_score(yte, p, zero_division=0),
    })
thr_df = pd.DataFrame(rows)
display(thr_df)

prec, rec, ths = precision_recall_curve(yte, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec)
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('PR Curve')
plt.grid(True, alpha=0.3)
plt.show()


## 5. QuickBaseline 对比（无 PCA）


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xtr_s = pd.DataFrame(scaler.fit_transform(Xtr), columns=Xtr.columns, index=Xtr.index)
Xte_s = pd.DataFrame(scaler.transform(Xte), columns=Xte.columns, index=Xte.index)

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_s, ytr, Xte_s, yte, task_type='classification')
comparison = baseline.compare_models(results, top_n=6)
comparison


## 6. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

best = comparison.iloc[0]['Model']
pred_best = results[best]['predictions']
out = Path('outputs/04_breast_cancer'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred_best)), 'target': yte.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred_best, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 7. 检查清单
- [x] 分层划分
- [x] 缩放 + PCA 管线
- [x] 阈值-召回权衡与 PR 曲线
- [ ] 按假阴性/假阳性业务成本选最终阈值
- [ ] 校准概率（calibration）后再定阈值
